# 01 · 从六个空模块搭出一个会修代码的 MiniCodex

`task_list.py` 应当按 `status` 筛选任务，现在却读错字段。我们从空模块开始，写出六个彼此连接的组件，让 Agent 读文件、发现失败、修改代码、重跑测试，最后由控制器确认完成。

本章在本机执行自建练习程序，子进程拥有当前用户的操作系统权限，操作系统沙箱在后续专题实现。

需要 Python 3.12+ 和 Jupyter。已有 Notebook 环境可直接从头逐格运行。新环境可执行 `python -m pip install jupyterlab`，再运行 `python -m jupyter lab`。六个模块的实现只用 Python 标准库。

默认使用 `ScriptedModel` 代替模型，预先给出五轮输出和正确补丁。读文件、写文件、运行候选程序和检查结果都会真实发生。最后一节可选接入 DeepSeek，观察真实模型如何决定下一步。

### 在这本 Notebook 中操作

本章的全部 Agent 源码都在下面的代码格中，直接在这里阅读、修改和执行。Notebook 是唯一源码；运行末尾的导出格后，`code/mini_codex/minimal/` 中的 `.py` 文件会自动同步，作为生成物使用。

点击下一节表中的模块名可跳转到实现，也可以打开 JupyterLab 左侧的「目录」。源码格默认全部展开，需要专注某一模块时，使用 JupyterLab 原生的单元格源码折叠即可。

首次学习从头逐格运行。整本运行后修改模块，按“运行修改后的源码格 → [重新加载并组装](#reload-and-assemble) → 下一格完整修复实验”的顺序操作。早期模块示例用于首次构建，依赖缺陷程序等初始状态；要重跑这些示例，请先用「Restart Kernel and Run All Cells」恢复完整构建流程。


## 1. 先确定本章全部模块

先看完整边界，再按依赖顺序逐一填充。表中的接口都会在本章实现并使用。

| 模块 | 职责 | 主要输入与输出 |
| --- | --- | --- |
| [`executor.py`](#module-executor) | 启动候选 Python 程序，收集进程结果 | `Executor.run_python(workspace, path, payload)` → 退出码、stdout、stderr |
| [`verifier.py`](#module-verifier) | 使用固定输入和预期值判断候选程序 | `Verifier(workspace, executor, cases).check()` → 总体 `passed` 与逐例结果 |
| [`tools.py`](#module-tools) | 向模型描述工具，把调用分派到文件操作或测试 | `Tools(workspace, verifier).specs()` / `.execute(call)` → 工具描述 / 工具结果 |
| [`history.py`](#module-history) | 保存一轮任务的消息，拼接下一次请求 | `History(task).request(instructions, tools)` → 请求字典，另有追加输出、结果和反馈的方法 |
| [`model.py`](#module-model) | 提供统一采样接口，支持脚本与 DeepSeek | `Model.sample(request)` → Responses 格式的 `output` 列表 |
| [`agent.py`](#module-agent) | 编排采样、工具执行和完成检查 | `Agent(model, tools, verifier, instructions).run(task)` → 状态、步数、历史、最终检查 |

箭头表示对象调用或依赖关系。`Agent.run()` 为每次任务创建一个 `History`，各次运行拥有自己的消息列表。

```text
Agent ──→ Model
  ├────→ History
  ├────→ Tools ──→ Verifier ──→ Executor
  └─────────────→ Verifier
```

下面创建真实的 Python 包 `mini_codex_v1` 和独立的候选程序目录 `candidate`，先把六个 `.py` 文件留空。`__init__.py` 先保持为空，避免实现尚未写完时提前导入；六个模块完成后再写入完整的包入口。这一步只搭目录，模块会在后续单元格中获得实现。

In [1]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
import difflib
import importlib
import json
import sys

# 让重载总是读取刚编辑的源码，不生成临时字节码缓存。
sys.dont_write_bytecode = True
# 在仓库外运行副本时，请显式填写这个项目的绝对路径。
PROJECT_ROOT = None

lesson = TemporaryDirectory(prefix="mini codex chapter 01 ")
lesson_root = Path(lesson.name)
PACKAGE_DIR = lesson_root / "mini_codex_v1"
workspace = lesson_root / "candidate"
PACKAGE_DIR.mkdir()
workspace.mkdir()

MODULE_NAMES = ("executor", "verifier", "tools", "history", "model", "agent")
(PACKAGE_DIR / "__init__.py").touch()
for name in MODULE_NAMES:
    (PACKAGE_DIR / f"{name}.py").touch()

# 重跑本章时，重新加载这一份临时包。
for name in list(sys.modules):
    if name == "mini_codex_v1" or name.startswith("mini_codex_v1."):
        del sys.modules[name]
sys.path.insert(0, str(lesson_root))
importlib.invalidate_caches()

assert all(path.stat().st_size == 0 for path in PACKAGE_DIR.glob("*.py"))
print("本章临时目录", lesson_root)
for path in sorted(lesson_root.rglob("*")):
    print("  " + str(path.relative_to(lesson_root)) + ("/" if path.is_dir() else ""))
print("六个模块已创建，当前内容均为空。")


本章临时目录 /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b
  candidate/
  mini_codex_v1/
  mini_codex_v1/__init__.py
  mini_codex_v1/agent.py
  mini_codex_v1/executor.py
  mini_codex_v1/history.py
  mini_codex_v1/model.py
  mini_codex_v1/tools.py
  mini_codex_v1/verifier.py
六个模块已创建，当前内容均为空。


## 2. 写入待修复程序和任务

候选程序从标准输入读取 JSON，再把筛选结果作为 JSON 写到标准输出。数据字段是 `status`，函数却读取 `state`，因此按状态筛选时会丢失匹配项。

`workspace` 是传给对象的路径。执行器、验证器和工具都会通过参数接收它。练习文件随 `lesson` 临时目录对象释放而清理，重启内核后从头运行会创建新的一份。

In [2]:
BUGGY_SOURCE = '''import json
import sys


def filter_tasks(tasks, status=None):
    if status is None:
        return tasks
    return [task for task in tasks if task.get("state") == status]


if __name__ == "__main__":
    request = json.load(sys.stdin)
    print(json.dumps(filter_tasks(request["tasks"], request.get("status"))))
'''

TASK = "修复 task_list.py 的 status 筛选，保留 JSON stdin/stdout 接口。"
INSTRUCTIONS = "你在修复一个小程序。先读文件并运行测试，再精确修改。测试通过后说明完成。"
TASKS = [{"id": 1, "status": "todo"}, {"id": 2, "status": "done"}]

(workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
print(BUGGY_SOURCE)


import json
import sys


def filter_tasks(tasks, status=None):
    if status is None:
        return tasks
    return [task for task in tasks if task.get("state") == status]


if __name__ == "__main__":
    request = json.load(sys.stdin)
    print(json.dumps(filter_tasks(request["tasks"], request.get("status"))))



<a id="module-executor"></a>

## 3. 实现 Executor，让程序真正运行

`Executor` 接收工作目录、相对文件路径、输入字典和超时时间，在子进程中启动该文件。输入字典序列化为 JSON 后送入 stdin，返回值保存进程退出码、stdout 和 stderr。

这个模块处理进程执行与超时。接下来 `Verifier` 会解释 stdout 中的业务结果。

`%%writefile` 把单元格中展示的源码原样写入指定文件。路径上的引号让带空格的临时目录也能工作。首次构建时，每完成一个模块，紧接着就重新加载并调用它。完成整章后的修改实验，请使用开头说明的“源码格 → 重新加载并组装 → 完整修复实验”路径。

In [3]:
%%writefile "$PACKAGE_DIR/executor.py"
"""Run one Python program. A subprocess is not an operating-system sandbox."""

import json
from pathlib import Path
import subprocess
import sys


class Executor:
    def run_python(self, workspace: Path, path: str, payload: dict,
                   timeout: float = 2) -> dict:
        workspace = Path(workspace).resolve()
        relative = Path(path)
        target = (workspace / relative).resolve()
        if relative.is_absolute() or not target.is_relative_to(workspace):
            raise ValueError("Python 文件必须位于工作目录内，且使用相对路径")
        try:
            completed = subprocess.run(
                [sys.executable, "-I", str(target)],
                input=json.dumps(payload, ensure_ascii=False),
                cwd=workspace, env={}, capture_output=True,
                text=True, encoding="utf-8", timeout=timeout,
            )
        except subprocess.TimeoutExpired as error:
            # TimeoutExpired can carry bytes even when text=True.
            stdout = error.stdout or b""
            stderr = error.stderr or b""
            return {
                "exit_code": None,
                "stdout": stdout.decode("utf-8", errors="replace")
                if isinstance(stdout, bytes) else stdout,
                "stderr": stderr.decode("utf-8", errors="replace")
                if isinstance(stderr, bytes) else stderr,
                "error": f"Python 执行超过 {timeout} 秒",
                "timed_out": True,
            }
        return {"exit_code": completed.returncode,
                "stdout": completed.stdout, "stderr": completed.stderr}


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/executor.py


In [4]:
importlib.reload(importlib.import_module("mini_codex_v1.executor"))
from mini_codex_v1.executor import Executor

executor = Executor()
execution = executor.run_python(
    workspace, "task_list.py", {"tasks": TASKS, "status": "done"},
)
assert execution["exit_code"] == 0
assert json.loads(execution["stdout"]) == []
print(execution)
print("程序正常退出，但筛选 done 得到空列表，预期应包含 id=2。")


{'exit_code': 0, 'stdout': '[]\n', 'stderr': ''}
程序正常退出，但筛选 done 得到空列表，预期应包含 id=2。


<a id="module-verifier"></a>

## 4. 实现 Verifier，把运行结果变成判断

`Verifier` 保存固定的输入和预期结果，通过刚写好的 `Executor` 逐例运行 `task_list.py`。每项结果包含 `expected`、`actual`、进程信息和 `passed`，`check()` 再汇总为整体判断。

固定案例保存在控制器对象中。`run_tests` 工具和 Agent 最后的完成检查都调用这个对象，判断依据一致。构造时复制案例，避免外部修改原列表影响本次验证。

In [5]:
%%writefile "$PACKAGE_DIR/verifier.py"
"""Compare candidate output with fixed cases owned by the controller."""

from copy import deepcopy
import json
from pathlib import Path

from .executor import Executor


class Verifier:
    def __init__(self, workspace: Path, executor: Executor, cases: list):
        if not cases:
            raise ValueError("必须提供非空的固定测试案例")
        for case in cases:
            if not isinstance(case.get("name"), str) or not isinstance(case.get("input"), dict):
                raise ValueError("每个案例必须包含 name 字符串与 input 对象")
            if "expected" not in case:
                raise ValueError("每个案例必须包含 expected")
        self.workspace = Path(workspace).resolve()
        self.executor = executor
        self._cases = deepcopy(cases)

    def check(self) -> dict:
        results = [self._check_case(case) for case in self._cases]
        return {"passed": all(row["passed"] for row in results), "cases": results}

    def _check_case(self, case: dict) -> dict:
        result = {"name": case["name"], "expected": deepcopy(case["expected"]),
                  "actual": None, "passed": False, "exit_code": None}
        try:
            execution = self.executor.run_python(self.workspace, "task_list.py", case["input"])
            result.update(execution)
            if execution["exit_code"] == 0:
                result["actual"] = json.loads(execution["stdout"])
                result["passed"] = self._same_json(result["actual"], case["expected"])
        except (OSError, ValueError) as error:
            result["error"] = f"{type(error).__name__}: {error}"
        return result

    @staticmethod
    def _same_json(actual, expected) -> bool:
        # Python considers True == 1; JSON serialization preserves their types.
        options = {"sort_keys": True, "ensure_ascii": False, "allow_nan": False}
        return json.dumps(actual, **options) == json.dumps(expected, **options)


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/verifier.py


In [6]:
importlib.reload(importlib.import_module("mini_codex_v1.verifier"))
from mini_codex_v1.verifier import Verifier

CASES = [
    {"name": "筛选 done", "input": {"tasks": TASKS, "status": "done"}, "expected": [TASKS[1]]},
    {"name": "筛选 todo", "input": {"tasks": TASKS, "status": "todo"}, "expected": [TASKS[0]]},
    {"name": "不筛选", "input": {"tasks": TASKS}, "expected": TASKS},
    {"name": "没有匹配项", "input": {"tasks": TASKS, "status": "missing"}, "expected": []},
    {"name": "空列表", "input": {"tasks": [], "status": "done"}, "expected": []},
]

verifier = Verifier(workspace, executor, CASES)
baseline = verifier.check()
assert [row["passed"] for row in baseline["cases"]] == [False, False, True, True, True]
for row in baseline["cases"]:
    print("绿" if row["passed"] else "红", row["name"], "实际", row["actual"], "预期", row["expected"])
print("修复前共 2 红 3 绿。")


红 筛选 done 实际 [] 预期 [{'id': 2, 'status': 'done'}]
红 筛选 todo 实际 [] 预期 [{'id': 1, 'status': 'todo'}]
绿 不筛选 实际 [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}] 预期 [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}]
绿 没有匹配项 实际 [] 预期 []
绿 空列表 实际 [] 预期 []
修复前共 2 红 3 绿。


<a id="module-tools"></a>

## 5. 实现 Tools，把模型请求接到具体能力

模型通过工具描述了解名称和参数，再返回一个 `function_call`。`Tools.execute()` 解析其中的 JSON `arguments`，分派到 `read_file`、`replace_text` 或 `run_tests`。

读写工具接收相对路径，访问范围限制在传入的工作目录内。`replace_text` 要求 `old` 恰好出现一次，以便精确定位修改。`run_tests` 直接调用上一步的 `Verifier`。参数或文件操作出错时，`execute()` 返回带 `error` 的结果，供后续消息记录使用。

In [7]:
%%writefile "$PACKAGE_DIR/tools.py"
"""Three tools with explicit file and test responsibilities."""

from copy import deepcopy
import json
from pathlib import Path

from .verifier import Verifier


TOOL_SPECS = [
    {"type": "function", "name": "read_file", "description": "读取工作目录内的 UTF-8 文件",
     "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                    "required": ["path"], "additionalProperties": False}, "strict": True},
    {"type": "function", "name": "replace_text", "description": "精确替换文件中唯一出现的文本",
     "parameters": {"type": "object", "properties": {
         "path": {"type": "string"}, "old": {"type": "string"}, "new": {"type": "string"}},
         "required": ["path", "old", "new"], "additionalProperties": False}, "strict": True},
    {"type": "function", "name": "run_tests", "description": "运行控制器保存的固定测试",
     "parameters": {"type": "object", "properties": {}, "required": [],
                    "additionalProperties": False}, "strict": True},
]


class Tools:
    def __init__(self, workspace: Path, verifier: Verifier):
        self.workspace = Path(workspace).resolve()
        self.verifier = verifier

    def specs(self) -> list:
        return deepcopy(TOOL_SPECS)

    def execute(self, call: dict) -> dict:
        try:
            arguments = json.loads(call["arguments"])
            if not isinstance(arguments, dict):
                raise ValueError("工具参数必须是 JSON 对象")
            name = call["name"]
            if name == "read_file":
                return self.read_file(**arguments)
            if name == "replace_text":
                return self.replace_text(**arguments)
            if name == "run_tests":
                return self.run_tests(**arguments)
            raise ValueError(f"未知工具：{name}")
        except (KeyError, TypeError, ValueError, OSError) as error:
            return {"error": f"{type(error).__name__}: {error}"}

    def read_file(self, path: str) -> dict:
        return {"path": path, "content": self._path(path).read_text(encoding="utf-8")}

    def replace_text(self, path: str, old: str, new: str) -> dict:
        if not isinstance(old, str) or not old or not isinstance(new, str):
            raise ValueError("old 必须是非空字符串，new 必须是字符串")
        target = self._path(path)
        content = target.read_text(encoding="utf-8")
        if content.count(old) != 1:
            raise ValueError("old 必须在文件中恰好出现一次")
        target.write_text(content.replace(old, new, 1), encoding="utf-8")
        return {"path": path, "replacements": 1}

    def run_tests(self) -> dict:
        return self.verifier.check()

    def _path(self, path: str) -> Path:
        if not isinstance(path, str) or not path:
            raise ValueError("path 必须是非空相对路径")
        relative = Path(path)
        target = (self.workspace / relative).resolve()
        if relative.is_absolute() or not target.is_relative_to(self.workspace):
            raise ValueError("文件必须位于工作目录内，且使用相对路径")
        return target


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/tools.py


In [8]:
importlib.reload(importlib.import_module("mini_codex_v1.tools"))
from mini_codex_v1.tools import Tools

tools = Tools(workspace, verifier)
READ_CALL = {
    "type": "function_call", "call_id": "read-1", "name": "read_file",
    "arguments": json.dumps({"path": "task_list.py"}),
}
observation = tools.execute(READ_CALL)
assert observation["content"] == BUGGY_SOURCE
print("提供给模型的工具", [spec["name"] for spec in tools.specs()])
print("读取结果字段", list(observation))

bad_patch = tools.execute({
    "type": "function_call", "call_id": "patch-demo", "name": "replace_text",
    "arguments": json.dumps({"path": "task_list.py", "old": "missing text", "new": "replacement"}),
})
assert "error" in bad_patch
assert tools.read_file("task_list.py")["content"] == BUGGY_SOURCE
print("无法定位的修改请求", bad_patch)


提供给模型的工具 ['read_file', 'replace_text', 'run_tests']
读取结果字段 ['path', 'content']
无法定位的修改请求 {'error': 'ValueError: old 必须在文件中恰好出现一次'}


<a id="module-history"></a>

## 6. 实现 History，让下一轮看到工具结果

一次请求包含 `instructions`、`input` 和 `tools`。`History` 保存 `input` 的消息列表，开始时只有任务。模型返回完整 `output` 后，先用 `add_output()` 追加，再用 `add_tool_result()` 记录实际执行结果。

调用与结果共享一个 `call_id`。例如 `read-1` 的结果应对应 `read-1` 的读取请求。`feedback()` 追加控制器反馈，`snapshot()` 返回当前历史的副本。下面先手动完成一次往返，随后交给 Agent 自动编排。

In [9]:
%%writefile "$PACKAGE_DIR/history.py"
"""Own the Responses message list and serialize matching tool results."""

from copy import deepcopy
import json


class History:
    def __init__(self, task: str):
        self._items = [{"role": "user", "content": task}]

    def request(self, instructions: str, tools: list) -> dict:
        return {"instructions": instructions, "input": self.snapshot(),
                "tools": deepcopy(tools)}

    def add_output(self, items: list) -> None:
        self._items.extend(deepcopy(items))

    def add_tool_result(self, call_id: str, result: dict) -> None:
        self._items.append({"type": "function_call_output", "call_id": call_id,
                            "output": json.dumps(result, ensure_ascii=False)})

    def feedback(self, text: str) -> None:
        self._items.append({"role": "user", "content": text})

    def snapshot(self) -> list:
        return deepcopy(self._items)


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/history.py


In [10]:
importlib.reload(importlib.import_module("mini_codex_v1.history"))
from mini_codex_v1.history import History

history_demo = History(TASK)
history_demo.add_output([READ_CALL])
history_demo.add_tool_result(READ_CALL["call_id"], observation)
next_request = history_demo.request(INSTRUCTIONS, tools.specs())
read_pair = next_request["input"][-2:]

assert read_pair[0]["call_id"] == read_pair[1]["call_id"] == "read-1"
assert json.loads(read_pair[1]["output"])["content"] == BUGGY_SOURCE
print("下一次请求的字段", list(next_request))
print("消息顺序", [(item.get("type", item.get("role")), item.get("call_id"))
                   for item in next_request["input"]])
print("下一次请求已包含读到的文件内容。")


下一次请求的字段 ['instructions', 'input', 'tools']
消息顺序 [('user', None), ('function_call', 'read-1'), ('function_call_output', 'read-1')]
下一次请求已包含读到的文件内容。


<a id="module-model"></a>

## 7. 实现 Model，统一模型的输入和输出

`Model` 用 `Protocol` 声明一个接口。任何提供 `sample(request)` 方法、返回 `output` 列表的对象都能接入。Agent 只依赖这个方法。

`ScriptedModel` 按顺序返回预设输出，并在 `requests` 中保存实际收到的每次请求。这使我们能够检查下一轮究竟看到了什么。脚本耗尽后返回普通消息，文件操作仍由 `Tools` 执行。

同一个模块也提供 `DeepSeekModel`。它把相同请求通过 HTTP 发给 Responses API，并返回完整 `output` 列表。先顺着 `sample()` 的输入与返回值阅读即可，HTTP 请求字段在后续 API 专题深入。定义这个类和导入模块都不会发起请求，最后一节才通过开关选择它。

In [11]:
%%writefile "$PACKAGE_DIR/model.py"
"""Replace the model without changing the loop, tools, or history."""

from copy import deepcopy
import json
from pathlib import Path
from typing import Protocol
import urllib.error
import urllib.request


class Model(Protocol):
    def sample(self, request: dict) -> list[dict]: ...


class ScriptedModel:
    """Replay explicit outputs; exhausting a script never changes any file."""

    def __init__(self, script: list):
        self.script = deepcopy(script)
        self.requests = []

    def sample(self, request: dict) -> list[dict]:
        index = len(self.requests)
        self.requests.append(deepcopy(request))
        if index < len(self.script):
            return deepcopy(self.script[index])
        return [{"type": "message", "role": "assistant", "content": [
            {"type": "output_text", "text": "脚本已结束。"}]}]


class DeepSeekModel:
    def __init__(self, api_key: str, model: str):
        if not api_key or not model:
            raise ValueError("必须提供 API key 和模型名称")
        self._api_key = api_key
        self.model = model

    @classmethod
    def from_env_file(cls, path: Path):
        settings = {}
        for raw_line in Path(path).read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            key, separator, value = line.partition("=")
            if not separator:
                raise ValueError("配置行必须使用 KEY=VALUE 格式")
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
                value = value[1:-1]
            settings[key.strip()] = value
        if settings.get("OPENAI_BASE_URL") != "https://api.deepseek.com":
            raise ValueError("本章要求 OPENAI_BASE_URL=https://api.deepseek.com")
        return cls(settings.get("OPENAI_API_KEY", ""), settings.get("OPENAI_MODEL", ""))

    def sample(self, request: dict) -> list[dict]:
        payload = {**deepcopy(request), "model": self.model,
                   "reasoning": {"effort": "none"}, "max_output_tokens": 2048,
                   "stream": False}
        http_request = urllib.request.Request(
            "https://api.deepseek.com/responses",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Authorization": f"Bearer {self._api_key}",
                     "Content-Type": "application/json"},
        )
        try:
            with urllib.request.urlopen(http_request, timeout=60) as response:
                body = json.load(response)
        except urllib.error.HTTPError as error:
            error.close()
            raise RuntimeError(f"DeepSeek 请求失败，HTTP {error.code}") from None
        except urllib.error.URLError:
            raise RuntimeError("无法连接 DeepSeek") from None
        if body.get("status") != "completed":
            raise RuntimeError("DeepSeek 响应未完成")
        output = body.get("output")
        if not isinstance(output, list) or not all(isinstance(item, dict) for item in output):
            raise RuntimeError("DeepSeek output 必须是对象列表")
        return deepcopy(output)


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/model.py


In [12]:
importlib.reload(importlib.import_module("mini_codex_v1.model"))
from mini_codex_v1.model import Model, ScriptedModel, DeepSeekModel

FINAL_MESSAGE = {"type": "message", "role": "assistant", "content": [
    {"type": "output_text", "text": "已修复 status 筛选。"},
]}
SCRIPT = [
    [READ_CALL],
    [{"type": "function_call", "call_id": "test-before", "name": "run_tests", "arguments": "{}"}],
    [{"type": "function_call", "call_id": "patch-1", "name": "replace_text",
      "arguments": json.dumps({"path": "task_list.py", "old": 'task.get("state")',
                               "new": 'task.get("status")'})}],
    [{"type": "function_call", "call_id": "test-after", "name": "run_tests", "arguments": "{}"}],
    [FINAL_MESSAGE],
]

model_demo = ScriptedModel(SCRIPT)
first_output = model_demo.sample(History(TASK).request(INSTRUCTIONS, tools.specs()))
assert first_output == [READ_CALL]
assert len(model_demo.requests) == 1
print("五轮预设输出", [items[0].get("name", "完成消息") for items in SCRIPT])
print("第一次采样返回", first_output)
print("当前文件仍是缺陷版本", tools.read_file("task_list.py")["content"] == BUGGY_SOURCE)


五轮预设输出 ['read_file', 'run_tests', 'replace_text', 'run_tests', '完成消息']
第一次采样返回 [{'type': 'function_call', 'call_id': 'read-1', 'name': 'read_file', 'arguments': '{"path": "task_list.py"}'}]
当前文件仍是缺陷版本 True


<a id="module-agent"></a>

## 8. 实现 Agent，只编排这些对象

现在把已经能单独工作的对象连接起来。`run()` 为本次任务创建 `History`，按预算推进循环。`_step()` 请求模型、记录输出，并决定执行工具还是检查完成。`_execute_tools()` 把执行结果交给历史，`_verify_completion()` 调用独立验证器。`_result()` 整理返回值。

本章约定没有工具调用的一轮表示模型请求结束。控制器随后重新运行全部案例，只有通过才返回 `passed`。失败时把验证结果加入历史，继续下一轮。每次 `sample()` 消耗一步，耗尽预算返回 `step_limit`，同时保留当时真实的验证结果。

顺着下面的调用看，HTTP 交给 `Model`，工具参数交给 `Tools`，进程执行交给 `Executor`，预期值比较交给 `Verifier`。

In [13]:
%%writefile "$PACKAGE_DIR/agent.py"
"""Coordinate model, history, tools and independent completion checks."""

import json

from .history import History
from .model import Model
from .tools import Tools
from .verifier import Verifier


class Agent:
    def __init__(self, model: Model, tools: Tools, verifier: Verifier, instructions: str):
        self.model = model
        self.tools = tools
        self.verifier = verifier
        self.instructions = instructions

    def run(self, task: str, max_steps: int = 8) -> dict:
        if type(max_steps) is not int or max_steps < 0:
            raise ValueError("max_steps 必须是非负整数")
        history = History(task)
        for step in range(1, max_steps + 1):
            verification = self._step(history)
            if verification is not None and verification["passed"]:
                return self._result("passed", step, history, verification)
        return self._result("step_limit", max_steps, history, self.verifier.check())

    def _step(self, history: History):
        output = self.model.sample(history.request(self.instructions, self.tools.specs()))
        if not output or not isinstance(output, list) or not all(isinstance(i, dict) for i in output):
            raise ValueError("模型必须返回非空 output 对象列表")
        history.add_output(output)
        calls = [item for item in output if item.get("type") == "function_call"]
        if calls:
            self._execute_tools(history, calls)
            return None
        # Teaching simplification: output without tool calls requests completion.
        return self._verify_completion(history)

    def _execute_tools(self, history: History, calls: list) -> None:
        for call in calls:
            call_id = call.get("call_id")
            if not isinstance(call_id, str) or not call_id:
                raise ValueError("工具调用必须包含非空 call_id")
            history.add_tool_result(call_id, self.tools.execute(call))

    def _verify_completion(self, history: History) -> dict:
        verification = self.verifier.check()
        if not verification["passed"]:
            history.feedback("控制器重跑测试仍失败，请继续修复。" +
                             json.dumps(verification, ensure_ascii=False))
        return verification

    @staticmethod
    def _result(status: str, steps: int, history: History, verification: dict) -> dict:
        return {"status": status, "steps": steps, "history": history.snapshot(),
                "verification": verification}


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/agent.py


In [14]:
importlib.reload(importlib.import_module("mini_codex_v1.agent"))
from mini_codex_v1.agent import Agent

one_step_model = ScriptedModel([[READ_CALL]])
one_step_agent = Agent(one_step_model, tools, verifier, INSTRUCTIONS)
one_step = one_step_agent.run(TASK, max_steps=1)
assert one_step["status"] == "step_limit"
assert one_step["history"][-1]["call_id"] == "read-1"
assert json.loads(one_step["history"][-1]["output"])["content"] == BUGGY_SOURCE
print("一次编排的状态", one_step["status"])
print("历史中的消息类型", [item.get("type", item.get("role")) for item in one_step["history"]])


一次编排的状态 step_limit
历史中的消息类型 ['user', 'function_call', 'function_call_output']


### 补齐 Python 包入口

六个模块都已经可以运行，现在写入 `__init__.py`，把本章的公开类集中到包入口。它只声明导入关系；下面保留完整内容，后续也会与六个模块一起导出。


In [15]:
%%writefile "$PACKAGE_DIR/__init__.py"
"""The six small components introduced in chapter 1."""

from .agent import Agent
from .executor import Executor
from .history import History
from .model import DeepSeekModel, Model, ScriptedModel
from .tools import Tools
from .verifier import Verifier

__all__ = ["Agent", "Executor", "History", "Model", "ScriptedModel",
           "DeepSeekModel", "Tools", "Verifier"]


Overwriting /var/folders/lj/bgmvywdd6l50bl6w6p5kmc6r0000gn/T/mini codex chapter 01 2g9ga52b/mini_codex_v1/__init__.py


## 9. 组装完整的 MiniCodex，完成五轮修复

六个模块和包入口都已有实现。下面显式创建执行器、验证器、工具、模型和 Agent，`History` 在 `run()` 内创建。它们处理同一个候选目录和同一份固定案例。

<a id="reload-and-assemble"></a>

### 重新加载并组装

Python 会缓存已导入的模块。每次修改并执行源码格后，运行下面这一格：先按依赖顺序重新加载六个模块，再加载包入口、重新绑定本 Notebook 使用的类名，并创建新的对象。旧对象不会自动变成新实现，因此每次都重新组装。

此格会恢复缺陷程序和全新的五轮脚本，便于重复实验。接着运行下一格：第三轮真正修改文件，第四轮测试，最后一轮请求结束并触发控制器重跑。


In [16]:
importlib.invalidate_caches()
for name in MODULE_NAMES:
    importlib.reload(importlib.import_module(f"mini_codex_v1.{name}"))
importlib.reload(importlib.import_module("mini_codex_v1"))
from mini_codex_v1 import Agent, Executor, History, Model, ScriptedModel, DeepSeekModel, Tools, Verifier

(workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
executor = Executor()
verifier = Verifier(workspace, executor, CASES)
tools = Tools(workspace, verifier)
model = ScriptedModel(SCRIPT)
agent = Agent(model, tools, verifier, INSTRUCTIONS)
print("已重新加载六个模块和包入口，并创建本轮的全新对象。")


已重新加载六个模块和包入口，并创建本轮的全新对象。


In [17]:
result = agent.run(TASK, max_steps=8)
fixed_source = tools.read_file("task_list.py")["content"]
assert result["status"] == "passed" and result["steps"] == 5
assert len(result["verification"]["cases"]) == 5
assert all(row["passed"] for row in result["verification"]["cases"])
assert fixed_source == BUGGY_SOURCE.replace('task.get("state")', 'task.get("status")')

print("结束状态", result["status"], "模型请求次数", result["steps"])
for row in result["verification"]["cases"]:
    print("绿" if row["passed"] else "红", row["name"], "实际", row["actual"])
print("控制器最终得到 5 绿。")
print("".join(difflib.unified_diff(
    BUGGY_SOURCE.splitlines(keepends=True), fixed_source.splitlines(keepends=True),
    fromfile="修改前/task_list.py", tofile="修改后/task_list.py",
)))


结束状态 passed 模型请求次数 5
绿 筛选 done 实际 [{'id': 2, 'status': 'done'}]
绿 筛选 todo 实际 [{'id': 1, 'status': 'todo'}]
绿 不筛选 实际 [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}]
绿 没有匹配项 实际 []
绿 空列表 实际 []
控制器最终得到 5 绿。
--- 修改前/task_list.py
+++ 修改后/task_list.py
@@ -5,7 +5,7 @@
 def filter_tasks(tasks, status=None):
     if status is None:
         return tasks
-    return [task for task in tasks if task.get("state") == status]
+    return [task for task in tasks if task.get("status") == status]
 
 
 if __name__ == "__main__":



### 检查工具结果是否进入下一轮

第二次模型请求应包含第一轮读到的原文，第三次请求应包含修复前的失败结果，最后一次请求应包含修复后的通过结果。所有调用和结果还应当通过 `call_id` 一一对应。

这些检查直接查看 `ScriptedModel.requests` 中记录的实际输入。完成消息之后，Agent 会再次调用验证器，最终结果保存在 `result["verification"]`。

In [18]:
read_pair = model.requests[1]["input"][-2:]
assert read_pair[0] == READ_CALL
assert read_pair[1]["type"] == "function_call_output"
assert read_pair[0]["call_id"] == read_pair[1]["call_id"]
assert json.loads(read_pair[1]["output"])["content"] == BUGGY_SOURCE

failed_tests = json.loads(model.requests[2]["input"][-1]["output"])
passed_tests = json.loads(model.requests[4]["input"][-1]["output"])
assert [row["passed"] for row in failed_tests["cases"]] == [False, False, True, True, True]
assert passed_tests["passed"]

call_ids = [item["call_id"] for item in result["history"] if item.get("type") == "function_call"]
result_ids = [item["call_id"] for item in result["history"] if item.get("type") == "function_call_output"]
assert call_ids == result_ids == ["read-1", "test-before", "patch-1", "test-after"]
assert model.requests[-1]["input"] == result["history"][:-1]

print("第二次请求已收到原文", json.loads(read_pair[1]["output"])["content"] == BUGGY_SOURCE)
print("第三次请求收到的测试", [(row["name"], row["passed"]) for row in failed_tests["cases"]])
print("第五次请求收到的测试", [(row["name"], row["passed"]) for row in passed_tests["cases"]])
print("配对成功的 call_id", call_ids)


第二次请求已收到原文 True
第三次请求收到的测试 [('筛选 done', False), ('筛选 todo', False), ('不筛选', True), ('没有匹配项', True), ('空列表', True)]
第五次请求收到的测试 [('筛选 done', True), ('筛选 todo', True), ('不筛选', True), ('没有匹配项', True), ('空列表', True)]
配对成功的 call_id ['read-1', 'test-before', 'patch-1', 'test-after']


## 10. 用反例检查结束条件

### 模型直接声称完成

在另一个候选目录里恢复缺陷程序，让模型连续两轮返回完成消息。验证器会发现筛选仍然失败，Agent 把失败结果交给下一次采样。两步后预算耗尽，返回 `step_limit`。

In [19]:
claim_workspace = lesson_root / "claim_candidate"
claim_workspace.mkdir(exist_ok=True)
(claim_workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
claim_verifier = Verifier(claim_workspace, Executor(), CASES)
claim_tools = Tools(claim_workspace, claim_verifier)
claim_model = ScriptedModel([[FINAL_MESSAGE], [FINAL_MESSAGE]])
claim_agent = Agent(claim_model, claim_tools, claim_verifier, INSTRUCTIONS)

false_claim = claim_agent.run(TASK, max_steps=2)
assert false_claim["status"] == "step_limit"
assert not false_claim["verification"]["passed"]
assert "控制器重跑测试仍失败" in claim_model.requests[1]["input"][-1]["content"]
assert claim_tools.read_file("task_list.py")["content"] == BUGGY_SOURCE
print("声称完成后的状态", false_claim["status"])
print("第二次请求收到失败反馈", "控制器重跑测试仍失败" in claim_model.requests[1]["input"][-1]["content"])
print("仍然失败的案例", [row["name"] for row in false_claim["verification"]["cases"] if not row["passed"]])


声称完成后的状态 step_limit
第二次请求收到失败反馈 True
仍然失败的案例 ['筛选 done', '筛选 todo']


### 正常脚本只有两步预算

再开一个候选目录，使用完整五轮脚本，把预算降为两步。它只能完成读取和首次测试，补丁位于第三轮，因此文件会保持缺陷版本。返回的 `verification` 记录停止时实际跑出的测试结果。

In [20]:
limited_workspace = lesson_root / "limited_candidate"
limited_workspace.mkdir(exist_ok=True)
(limited_workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
limited_verifier = Verifier(limited_workspace, Executor(), CASES)
limited_tools = Tools(limited_workspace, limited_verifier)
limited_model = ScriptedModel(SCRIPT)
limited_agent = Agent(limited_model, limited_tools, limited_verifier, INSTRUCTIONS)

limited = limited_agent.run(TASK, max_steps=2)
assert limited["status"] == "step_limit" and limited["steps"] == 2
assert len(limited_model.requests) == 2
assert not limited["verification"]["passed"]
assert limited_tools.read_file("task_list.py")["content"] == BUGGY_SOURCE
print({"status": limited["status"], "steps": limited["steps"],
       "tests_passed": limited["verification"]["passed"],
       "file_changed": limited_tools.read_file("task_list.py")["content"] != BUGGY_SOURCE})


{'status': 'step_limit', 'steps': 2, 'tests_passed': False, 'file_changed': False}


## 11. 改一个参数，先预测再运行

把 `practice_steps` 依次设为 `2`、`4`、`5`，预测文件是否已经修改、测试是否通过、Agent 返回什么状态。第四轮已运行修复后的测试，第五轮才出现完成请求。

接着把 `practice_old` 改为文件中不存在的文本，并恢复五步预算。观察补丁错误进入哪一轮的输入，最终为什么仍然失败。此格每次都会恢复缺陷文件并创建新模型，便于重复尝试。

In [21]:
practice_steps = 5
practice_old = 'task.get("state")'

practice_workspace = lesson_root / "practice_candidate"
practice_workspace.mkdir(exist_ok=True)
(practice_workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
practice_script = deepcopy(SCRIPT)
practice_script[2][0]["arguments"] = json.dumps({
    "path": "task_list.py", "old": practice_old, "new": 'task.get("status")',
})
practice_verifier = Verifier(practice_workspace, Executor(), CASES)
practice_tools = Tools(practice_workspace, practice_verifier)
practice_model = ScriptedModel(practice_script)
practice_agent = Agent(practice_model, practice_tools, practice_verifier, INSTRUCTIONS)

practice = practice_agent.run(TASK, max_steps=practice_steps)
print({"status": practice["status"], "steps": practice["steps"],
       "tests_passed": practice["verification"]["passed"],
       "file_changed": practice_tools.read_file("task_list.py")["content"] != BUGGY_SOURCE})
if len(practice_model.requests) >= 4:
    patch_result = practice_model.requests[3]["input"][-1]
    print("第四次请求收到的补丁结果", json.loads(patch_result["output"]))


{'status': 'passed', 'steps': 5, 'tests_passed': True, 'file_changed': True}
第四次请求收到的补丁结果 {'path': 'task_list.py', 'replacements': 1}


## 12. 可选，接入真实 DeepSeek

`DeepSeekModel` 已经定义在 `model.py` 中。这里创建该对象，再按相同依赖关系组装一个 Agent。真实模型将自行选择工具和修改内容。

使用项目根目录的私有 `.env.local`，参考 `code/.env.example` 填写 `OPENAI_API_KEY`、`OPENAI_MODEL` 和 `OPENAI_BASE_URL=https://api.deepseek.com`。此格从当前目录或父目录定位配置，也可手动设置 `ENV_FILE`。密钥保留在私有文件中。

`RUN_LIVE` 默认关闭。设为 `True` 后会读取配置，向 `https://api.deepseek.com/responses` 发起真实请求并产生 API 费用。本章适配器使用非流式响应，关闭思考模式。

In [22]:
RUN_LIVE = False

if RUN_LIVE:
    ENV_FILE = Path.cwd() / ".env.local"
    if not ENV_FILE.is_file():
        ENV_FILE = Path.cwd().parent / ".env.local"

    live_workspace = lesson_root / "live_candidate"
    live_workspace.mkdir(exist_ok=True)
    (live_workspace / "task_list.py").write_text(BUGGY_SOURCE, encoding="utf-8")
    live_verifier = Verifier(live_workspace, Executor(), CASES)
    live_tools = Tools(live_workspace, live_verifier)
    live_model = DeepSeekModel.from_env_file(ENV_FILE)
    live_agent = Agent(live_model, live_tools, live_verifier, INSTRUCTIONS)

    live = live_agent.run(TASK, max_steps=8)
    print({"status": live["status"], "steps": live["steps"],
           "tests": [(row["name"], row["passed"]) for row in live["verification"]["cases"]]})
    print("".join(difflib.unified_diff(
        BUGGY_SOURCE.splitlines(keepends=True),
        live_tools.read_file("task_list.py")["content"].splitlines(keepends=True),
        fromfile="修改前/task_list.py", tofile="修改后/task_list.py",
    )))
else:
    print("RUN_LIVE=False。本次运行使用 ScriptedModel，未读取私有配置或调用 DeepSeek。")


RUN_LIVE=False。本次运行使用 ScriptedModel，未读取私有配置或调用 DeepSeek。


## 13. 把本次执行的源码导出为 Python 包

下面把临时包中**已经执行源码格、写入磁盘的七个文件**导出到 `code/mini_codex/minimal/`。Notebook 仍是唯一源码，请在上面的代码格中修改，再运行对应源码格和这个导出格；不要手动维护另一份 `.py`。这里不读取尚未保存的 Notebook，因此本次编辑只要已执行，就会导出新内容。

导出工具 `code/notebook_tools.py` 只负责同步文件。它要求当前目录或父目录是本项目根目录；若在仓库外运行副本，请先在开头显式设置 `PROJECT_ROOT`。从命令行导出时应先保存 Notebook，再在项目根目录运行 `python code/notebook_tools.py export notebooks/01-loop.ipynb`。

可选：先保存并关闭 Notebook，运行 `python code/notebook_tools.py focus notebooks/01-loop.ipynb --modules executor` 后重新打开，只展开执行器源码；不指定 `--modules` 则恢复全部展开。折叠只改变显示，完整源码始终保留。


In [23]:
def is_project_root(path):
    return (path / "code/notebook_tools.py").is_file() and (path / "README.md").is_file()

if PROJECT_ROOT is None:
    PROJECT_ROOT = next((path for path in (Path.cwd(), Path.cwd().parent)
                         if is_project_root(path)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("未定位到本项目，请在开头显式设置 PROJECT_ROOT 后重跑此格。")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if not is_project_root(PROJECT_ROOT):
    raise FileNotFoundError(f"PROJECT_ROOT 缺少 code/notebook_tools.py 或 README.md：{PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "code"))
try:
    from notebook_tools import export_sources
finally:
    sys.path.pop(0)

EXPORT_DIR = PROJECT_ROOT / "code/mini_codex/minimal"
executed_sources = {
    name: (PACKAGE_DIR / f"{name}.py").read_text(encoding="utf-8")
    for name in (*MODULE_NAMES, "__init__")
}
export_report = export_sources(executed_sources, EXPORT_DIR)
print("已同步本次执行的七个源码文件到", EXPORT_DIR)
print("本次写入", export_report["written"], "内容一致", export_report["unchanged"])


已同步本次执行的七个源码文件到 /Users/te/Documents/Codex/2026-11-12/ban/.tmp/chapter01-review-20261115T082958Z/round-02/export/clean public project with spaces/code/mini_codex/minimal
本次写入 ['executor', 'verifier', 'tools', 'history', 'model', 'agent', '__init__'] 内容一致 []


## 回到模块边界，对照 Codex

现在临时包中的六个 `.py` 文件组成了本章的 MiniCodex。上一格已把这六个模块和包入口导出到 `code/mini_codex/minimal/`，后续可作为普通 Python 包使用；需要修改时回到本 Notebook 的源码格，再运行导出。六模块的划分是本教材的教学设计，目的是把执行、判断、工具、历史、模型和编排的职责分别展开。

这次默认运行使用了预设补丁，验证了模块连接、真实文件修改和五个输入的结果。真实模型的修复过程可以通过上一节单独观察。

Codex 还需要处理流式响应、取消、并发和权限。以下链接固定到提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`，可沿着提交、执行一轮、接收工具结果这条线阅读。

- `submission_loop` 接收并分派提交。[对应源码](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/handlers.rs#L474-L538)
- `run_turn` 推进一轮任务。[对应源码](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn.rs#L217-L241)
- `drain_in_flight` 收集完成的工具项并写入会话历史。[对应源码](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn.rs#L2851-L2879)

真实模型适配器的字段与端点可对照 DeepSeek 官方说明。[Responses API 使用指南](https://api-docs.deepseek.com/guides/responses_api/) · [创建响应接口](https://api-docs.deepseek.com/api/create-response/)